# LangSmith Tracing — Observability for LangGraph Agents
## Every Run, Logged — LangSmith Trace Anatomy
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/73-langsmith-tracing/langsmith_tracing_workbook.ipynb)

Instruments a LangGraph Q&A agent with LangSmith tracing via @traceable. Requires LANGSMITH_API_KEY and LANGSMITH_TRACING=true env vars. Shows how traces, spans, and metadata appear in the LangSmith UI.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why observability matters; LangSmith traces vs logs |
| 2 | **Setup** — LANGSMITH_API_KEY + LANGSMITH_TRACING=true; auto-instrumentation |
| 3 | **@traceable** — Decorator turns any function into a traced span |
| 4 | **LangGraph Auto-Tracing** — LangGraph pipelines traced automatically when env vars are set |
| 5 | **Full Pipeline** — Traced Q&A graph with named spans per node |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "langsmith", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    # dotenv fallback: reads OPENAI_API_KEY from .env in the project root
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — What Is a Trace?

Observability for LLM apps has three levels:

| Level | What it captures | LangSmith term |
|-------|-----------------|----------------|
| **Trace** | One full invocation (user sends a question) | Run |
| **Span** | One step inside that run (single LLM call, one graph node) | Child Run |
| **Token** | Individual output token | Not stored per-token |

### Two ways to enable tracing

**1. Env-var auto-tracing** — set env vars and LangGraph instruments every node automatically. Zero code changes needed.

```bash
export LANGSMITH_API_KEY=lsv2_...
export LANGSMITH_TRACING=true
export LANGSMITH_PROJECT=my-project   # optional; defaults to "default"
```

**2. `@traceable` decorator** — wrap any function to create a named span. Works anywhere in your code.

### What LangSmith shows per trace

- **Latency** — wall-clock time per span
- **Token usage** — prompt + completion tokens per LLM call
- **Inputs / Outputs** — full data flowing in and out of each span
- **Tags + Metadata** — custom labels and key-value data you attach

In [ ]:
import os
from langsmith import traceable

LANGSMITH_KEY   = bool(os.environ.get("LANGSMITH_API_KEY"))
LANGSMITH_TRACE = os.environ.get("LANGSMITH_TRACING", "false").lower() == "true"

print(f"LANGSMITH_API_KEY set:  {LANGSMITH_KEY}")
print(f"LANGSMITH_TRACING:      {LANGSMITH_TRACE}")
if not LANGSMITH_KEY:
    print("\nRunning without LangSmith — code works, traces just won't be logged.")
    print("Set LANGSMITH_API_KEY + LANGSMITH_TRACING=true in .env to activate.")
else:
    print("\nTracing active — check smith.langchain.com after running.")

# @traceable wraps any function; name= controls how it appears in the trace tree
@traceable(name="my-first-span")
def greet(name: str) -> str:
    return f"Hello, {name}! This call is now a LangSmith span."

print(f"\nResult: {greet('LangSmith')}")
print("-> If tracing is on, 'my-first-span' is now visible in the LangSmith UI.")

## Part 2 — LangGraph Auto-Tracing

When `LANGSMITH_TRACING=true` is set, **LangGraph automatically instruments every node** as a child span.

The trace tree for a 3-node pipeline looks like:

```
Run: "LangGraph"                    <- the .invoke() call
  |- Span: "retrieve"               <- node 1
  |    '- Span: "ChatOpenAI"        <- LLM call inside node 1
  |- Span: "generate"               <- node 2
  |    '- Span: "ChatOpenAI"        <- LLM call inside node 2
  '- Span: "format_output"          <- node 3 (no LLM — no child span)
```

Each node span captures its **input state dict** and **output state dict** automatically. Full data visibility at zero instrumentation cost.

> This section assumes you've built LangGraph pipelines before. It shows what tracing adds — no new graph concepts are introduced here.

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

class QAState(TypedDict):
    question: str
    context:  str
    answer:   str

def retrieve(state: QAState) -> dict:
    # simulates a vector store lookup; replace with retriever.invoke() in production
    return {"context": f"[retrieved context for: {state['question']}]"}

def generate(state: QAState) -> dict:
    prompt = f"Context: {state['context']}\n\nAnswer briefly: {state['question']}"
    return {"answer": llm.invoke([HumanMessage(content=prompt)]).content.strip()}

def format_output(state: QAState) -> dict:
    # post-processing node — clean whitespace, truncate, add prefix; no LLM call
    return {"answer": state["answer"].strip()}

g = StateGraph(QAState)
g.add_node("retrieve",      retrieve)
g.add_node("generate",      generate)
g.add_node("format_output", format_output)
g.add_edge(START, "retrieve")
g.add_edge("retrieve",      "generate")
g.add_edge("generate",      "format_output")
g.add_edge("format_output", END)
app = g.compile()

QUESTIONS = [
    "What is LangGraph?",
    "How does RAGAS evaluate faithfulness?",
    "What is the purpose of a checkpointer in LangGraph?",
]

for q in QUESTIONS:
    r = app.invoke({"question": q, "context": "", "answer": ""})
    print(f"Q: {q}")
    print(f"A: {r['answer'][:120]}")
    print()

print("One trace per .invoke(). Nodes retrieve / generate / format_output each become a child span.")

## Part 3 — Custom Metadata and Tags

`@traceable` accepts optional arguments that control what appears in LangSmith:

| Argument | Type | Purpose |
|----------|------|---------|
| `name` | `str` | Display name in the trace tree |
| `tags` | `list[str]` | Labels for filtering (version, env, experiment) |
| `metadata` | `dict` | Key-value data attached to the span |
| `run_type` | `str` | `"llm"`, `"retriever"`, `"chain"`, `"tool"` |

### Practical tagging patterns

```python
# version + env
@traceable(tags=["v2", "production"])

# per-user attribution
@traceable(metadata={"user_id": "u123", "session_id": "s456"})

# experiment comparison
@traceable(tags=["experiment-prompt-v3"], metadata={"variant": "concise"})
```

### How nested spans work

When a `@traceable` function calls another `@traceable` function, the inner call becomes a **child span** automatically — no explicit linking required. LangSmith infers the parent-child relationship from the Python call stack.

In [ ]:
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

# run_type="retriever" shows a document icon in the LangSmith UI
@traceable(name="vector-retrieve", run_type="retriever", tags=["retrieval", "v1"])
def retrieve_docs(query: str) -> list[str]:
    return [f"[doc-1: context about '{query}']", f"[doc-2: more context about '{query}']"]

@traceable(name="llm-generate", run_type="llm", tags=["generation", "v1"])
def generate_answer(question: str, docs: list[str]) -> str:
    context = "\n".join(docs)
    return llm.invoke([
        HumanMessage(content=f"Context: {context}\n\nAnswer briefly: {question}")
    ]).content.strip()

# top-level span; child spans nest automatically when called from here
@traceable(name="rag-pipeline", tags=["rag", "v1"],
           metadata={"model": "gpt-5.4-nano", "retrieval_k": 2, "pipeline_version": "1.0"})
def run_rag(question: str, user_id: str = "anon") -> str:
    docs = retrieve_docs(question)        # child span: "vector-retrieve"
    return generate_answer(question, docs)  # child span: "llm-generate"

for uid, q in [("user_42", "What is LangGraph?"), ("user_99", "What does RAGAS measure?")]:
    ans = run_rag(q, user_id=uid)
    print(f"[{uid}] Q: {q}")
    print(f"         A: {ans[:100]}")
    print()

print("LangSmith trace: 'rag-pipeline' (parent) -> 'vector-retrieve' + 'llm-generate' (children)")

## Part 4 — Tracing Evaluation Loops

Wrapping an eval loop in `@traceable` lets you track score trends over time in LangSmith — no separate dashboard needed.

**The pattern:**
1. Outer `@traceable` = the full eval run (parent span, tagged by model + dataset version)
2. Inner `@traceable` per question = individual answer + score (child spans)
3. Scores as `metadata` = searchable per-question performance without reading raw outputs

Run evals twice — once per model — and compare in LangSmith by filtering on model tags. This replaces manual spreadsheet tracking with a searchable, version-controlled observability record.

In [ ]:
import numpy as np
from langsmith import traceable
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

eval_llm    = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

GOLDEN_SET = [
    {"question": "What is the capital of France?", "expected": "Paris"},
    {"question": "What is 2 + 2?",                 "expected": "4"},
    {"question": "Who wrote Romeo and Juliet?",     "expected": "Shakespeare"},
]

@traceable(name="agent-answer", run_type="llm", tags=["eval", "golden-set"])
def get_answer(question: str) -> str:
    return eval_llm.invoke([
        HumanMessage(content=f"Answer in one word or number: {question}")
    ]).content.strip()

@traceable(name="cosine-score", run_type="tool", tags=["eval"])
def cosine_score(expected: str, actual: str) -> float:
    ev = np.array(embed_model.embed_query(expected))
    av = np.array(embed_model.embed_query(actual))
    return float(np.dot(ev, av) / (np.linalg.norm(ev) * np.linalg.norm(av)))

# outer span = the full eval run; all inner @traceable calls nest under it
@traceable(name="eval-run-v1", tags=["eval", "golden-set-v1"],
           metadata={"n_questions": len(GOLDEN_SET), "threshold": 0.80, "model": "gpt-5.4-nano"})
def run_evaluation() -> dict:
    scores = []
    for row in GOLDEN_SET:
        actual = get_answer(row["question"])
        score  = cosine_score(row["expected"], actual)
        scores.append(score)
        print(f"  {'PASS' if score >= 0.80 else 'FAIL'} | {row['question']}")
        print(f"       expected={row['expected']:12s}  actual={actual:12s}  score={score:.3f}")
    avg = sum(scores) / len(scores)
    return {"avg_score": round(avg, 4), "pass_rate": round(sum(1 for s in scores if s >= 0.80) / len(scores), 2)}

print("Running traced evaluation loop...\n")
summary = run_evaluation()
print(f"\nSummary: avg_score={summary['avg_score']}  pass_rate={summary['pass_rate']:.0%}")
print("\nIn LangSmith: filter tag='eval' to see all eval runs. Each question is a child span.")

## Exercises

Work through these in order — each builds directly on a section above.

---

**Exercise 1 — Tag a LangGraph Run**

Take the compiled `app` from Part 2 and pass `config={"tags": ["exercise", "workshop-v1"]}` to `.invoke()`. Verify the run appears with that tag in LangSmith.

*Hint:* `app.invoke(state, config={"tags": ["exercise", "workshop-v1"]})`

---

**Exercise 2 — Nested Spans**

Write two `@traceable` functions:
- `fetch_context(query: str) -> str` — returns a fake context string; `run_type="retriever"`, `tags=["data"]`
- `respond(query: str) -> str` — calls `fetch_context()` inside it, then the LLM; `tags=["llm"]`

Call `respond("What is LangGraph?")`. Verify the trace shows `respond` as parent with `fetch_context` nested under it.

---

**Exercise 3 — Score as Span Metadata**

Modify `get_answer()` from Part 4 to accept an optional `langsmith_extra` keyword argument. After computing the cosine score, call `get_answer` again with `langsmith_extra={"metadata": {"score": ..., "passed": ...}}` to attach the score to the span.

*Hint:* `langsmith_extra` is intercepted by `@traceable` — it never reaches the function body.

---

**Exercise 4 — Multi-Model Comparison**

Run `run_evaluation()` twice: once with `gpt-4o-mini` and once with `gpt-4o` (if you have access). Tag each run with the model name. Compare average scores in LangSmith by filtering each model tag.

In [ ]:
# Exercise 1 — Tag a LangGraph Run
# app is compiled in Part 2 (re-run that cell first if kernel restarted)

for q in ["What is LangGraph?", "What does RAGAS measure?"]:
    r = app.invoke(
        {"question": q, "context": "", "answer": ""},
        config={"tags": ["exercise", "workshop-v1"]}
    )
    print(f"Q: {q}")
    print(f"A: {r['answer'][:100]}")
    print()
# Filter tag='exercise' in LangSmith to isolate these runs

In [ ]:
# Exercise 2 — Nested Spans (your code here)
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

# TODO: define fetch_context() with @traceable(run_type="retriever", tags=["data"])
# TODO: define respond() with @traceable(tags=["llm"]) that calls fetch_context() then the LLM

# result = respond("What is LangGraph?")
# print(result)

---
## Answer Key

Attempt the exercises before reading the answers below.

In [ ]:
# Answer 1 — Tag a LangGraph Run
for q in ["What is LangGraph?", "What does RAGAS measure?"]:
    r = app.invoke(
        {"question": q, "context": "", "answer": ""},
        config={"tags": ["exercise", "workshop-v1"]}
    )
    print(f"Q: {q} -> {r['answer'][:80]}")
# Filter by tag 'exercise' in LangSmith to see only these runs

In [ ]:
# Answer 2 — Nested Spans
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

@traceable(name="fetch-context", run_type="retriever", tags=["data"])
def fetch_context(query: str) -> str:
    return "[LangGraph is a framework for stateful, multi-actor LLM applications]"

@traceable(name="respond", run_type="llm", tags=["llm"])
def respond(query: str) -> str:
    ctx = fetch_context(query)  # nested call -> child span under "respond"
    return _llm.invoke([HumanMessage(content=f"Context: {ctx}\n\nAnswer briefly: {query}")]).content.strip()

print(respond("What is LangGraph?"))
# LangSmith: "respond" is parent; "fetch-context" is the child span beneath it

In [ ]:
# Answer 3 — Score as Span Metadata
import numpy as np
from langsmith import traceable
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

_llm    = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
_embeds = OpenAIEmbeddings(model="text-embedding-3-small")

@traceable(name="agent-answer-v2", run_type="llm", tags=["eval"])
def get_answer_meta(question: str, langsmith_extra: dict | None = None) -> str:
    # langsmith_extra is consumed by @traceable; function body never sees it
    return _llm.invoke([HumanMessage(content=f"Answer briefly: {question}")]).content.strip()

for row in [{"question": "Capital of France?", "expected": "Paris"},
            {"question": "What is 2+2?",        "expected": "4"}]:
    actual = get_answer_meta(row["question"])
    ev  = np.array(_embeds.embed_query(row["expected"]))
    av  = np.array(_embeds.embed_query(actual))
    score = float(np.dot(ev, av) / (np.linalg.norm(ev) * np.linalg.norm(av)))
    # pass score as metadata so it's searchable in LangSmith without reading outputs
    get_answer_meta(row["question"],
        langsmith_extra={"metadata": {"score": round(score, 4), "passed": score >= 0.80}})
    print(f"Q: {row['question']}  score={score:.3f}  passed={score >= 0.80}")

In [ ]:
# Answer 4 — Multi-Model Comparison
import numpy as np
from langsmith import traceable
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

_embeds = OpenAIEmbeddings(model="text-embedding-3-small")
GOLDEN  = [
    {"question": "Capital of France?", "expected": "Paris"},
    {"question": "Author of Hamlet?",  "expected": "Shakespeare"},
    {"question": "Square root of 64?", "expected": "8"},
]

def eval_model(model_name: str) -> float:
    _llm = ChatOpenAI(model=model_name, temperature=0)

    @traceable(name=f"eval-{model_name}", tags=["eval", model_name],
               metadata={"model": model_name, "n": len(GOLDEN)})
    def _run():
        scores = []
        for row in GOLDEN:
            actual = _llm.invoke([HumanMessage(content=f"One word: {row['question']}")]).content.strip()
            ev = np.array(_embeds.embed_query(row["expected"]))
            av = np.array(_embeds.embed_query(actual))
            scores.append(float(np.dot(ev, av) / (np.linalg.norm(ev) * np.linalg.norm(av))))
        return round(sum(scores)/len(scores), 4)

    return _run()

for model in ["gpt-5.4-nano"]:   # add another supported model to compare
    print(f"{model}: avg_cosine={eval_model(model)}")
print("\nFilter tag='eval' in LangSmith to compare runs by model.")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*